# Remote SSH Validation Notebook for Adeept RaspTank Pro

This notebook is for a safe, beginner-friendly first test of the refactored robot package when connected from a laptop using VS Code Remote SSH.

Goal:
- open the repo on the Raspberry Pi over SSH
- confirm the repo is on the Python path
- confirm the correct Jupyter kernel is active
- confirm `ipykernel` is installed
- instantiate the robot
- do a very small, low-power first movement test

IMPORTANT SAFETY WARNING:
- Do not run real movement unless the robot is physically secured on a stand or table.
- Use a very low speed.
- Keep the test very short.
- Stop immediately if anything acts unexpectedly.
- This notebook is intentionally cautious and should be treated as a first-pass validation tool.


In [ ]:
# Cell 1: Check the current working directory and confirm the repo is mounted on the Raspberry Pi.
# This is useful when using VS Code Remote SSH.

from pathlib import Path
import os

print('CWD =', Path.cwd())
print('HOME =', Path.home())

candidate_roots = [
    Path.cwd(),
    Path.cwd().parent,
    Path.home() / 'Adeept_RaspTankPro',
    Path('/workspace'),
]

for p in candidate_roots:
    if p.exists():
        print('candidate =', p, 'exists =', p.exists())
        if (p / 'rasptank').exists() or (p / 'server').exists():
            print('PROBABLE_REPO_ROOT =', p)


In [ ]:
# Cell 2: Make sure the repo is on sys.path.
# If you are already in the repo root, this usually works automatically.

import sys
from pathlib import Path

repo_root = None
for candidate in [
    Path.cwd(),
    Path.cwd().parent,
    Path.home() / 'Adeept_RaspTankPro',
    Path('/workspace'),
]:
    if (candidate / 'rasptank').exists() and (candidate / 'server').exists():
        repo_root = candidate
        break

if repo_root is not None:
    sys.path.insert(0, str(repo_root))
    print('Repo root inserted into sys.path:', repo_root)
else:
    print('Could not automatically find the repo root. Please open the repo folder first in VS Code Remote SSH.')


In [ ]:
# Cell 3: Confirm that the correct Python kernel is active and that ipykernel is installed.
# This is important when using VS Code Remote SSH and a remote Jupyter kernel.

import sys
import subprocess

print('Python executable =', sys.executable)
print('Python version =', sys.version)

try:
    import ipykernel
    print('IPYKERNEL_OK =', ipykernel.__version__)
except Exception as exc:
    print('IPYKERNEL_MISSING =', repr(exc))
    print('Installing ipykernel...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'ipykernel'])
    import ipykernel
    print('IPYKERNEL_OK_AFTER_INSTALL =', ipykernel.__version__)


In [ ]:
# Cell 4: Import the package and confirm the public API is exposed.
# This should work after the repo is on sys.path.

try:
    import rasptank
    print('IMPORT_OK: rasptank imported successfully')
    print('PACKAGE_EXPORTS:', [name for name in dir(rasptank) if not name.startswith('_')])
except Exception as exc:
    print('IMPORT_ERROR:', repr(exc))
    raise


In [ ]:
# Cell 5: Confirm the lower-level classes are available.

try:
    from rasptank import RaspTank
    from rasptank.hardware.motors import MotorController
    from rasptank.hardware.servos import ServoController
    from rasptank.hardware.ultrasonic import UltrasonicSensor
    from rasptank.hardware.leds import LEDStrip
    from rasptank.hardware.display import OledDisplay
    print('IMPORT_OK: all robot classes loaded')
except Exception as exc:
    print('IMPORT_ERROR:', repr(exc))
    raise


## Safety gate before real movement

Before you run the next cell, confirm this:
- the robot is physically secured
- it is elevated or clamped so it cannot roll away
- all wiring is checked
- you can stop power immediately
- you are running only a tiny movement test
- the speed is low (example: 10 to 20)
- the duration is tiny (example: 0.2 to 0.5 seconds)

If not, do not run the real movement cell.


In [ ]:
# Cell 6: SAFE FIRST MOTOR TEST
# This cell is disabled by default to protect the robot and the user.
# Set RUN_SAFE_TEST = True only when you are ready and the robot is securely mounted.

import time

RUN_SAFE_TEST = False

if RUN_SAFE_TEST:
    tank = RaspTank()
    try:
        print('Starting short low-power forward motion...')
        tank.forward(15)
        time.sleep(0.3)
        tank.stop()
        print('Motion test complete. Robot stopped.')
    finally:
        tank.close()
        print('Hardware closed cleanly.')
else:
    print('SAFE_TEST_DISABLED: set RUN_SAFE_TEST = True only after the robot is secured and safe to move.')


In [ ]:
# Cell 7: Optional tiny turn test
# This should remain disabled until the straight test is stable.

RUN_TURN_TEST = False

if RUN_TURN_TEST:
    tank = RaspTank()
    try:
        tank.turn_left(12)
        time.sleep(0.25)
        tank.stop()
        time.sleep(0.2)
        tank.turn_right(12)
        time.sleep(0.25)
        tank.stop()
        print('Turn test complete.')
    finally:
        tank.close()
else:
    print('TURN_TEST_DISABLED: keep this off until forward motion is verified and safe.')


## Diagnostics summary

This notebook is intentionally conservative. Its purpose is to make sure:
- the repo is present on the Raspberry Pi
- the correct Python kernel is active
- `ipykernel` is installed
- imports succeed
- the robot object can be instantiated
- the first real motion test remains tiny and safe

If all of the above succeed, then the next step is a low-power motion check in a secure physical setup.
